# tabx CSV EDA Notebook Demo

This notebook shows a practical daily workflow: load a CSV file in one call and continue analysis in pandas.

In [ ]:
from pathlib import Path
import pandas as pd
import tabx

In [ ]:
# Set your CSV path.
csv_path = Path('data/sample_sales.csv')

# If the file does not exist yet, create a small demo dataset.
if not csv_path.exists():
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    csv_path.write_text(
        'order_id,region,qty,unit_price,status,comment\n'
        '1,EMEA,2,120.5,paid,ok\n'
        '2,APAC,1,99.9,pending,NA\n'
        '3,EMEA,5,45.0,paid,priority\n'
        '4,AMER,3,80.0,cancelled,NULL\n'
        '5,EMEA,4,65.5,paid,ok\n'
        encoding='utf-8',
    )

csv_path

In [ ]:
# One-line UX: load CSV directly from a file path.
df = tabx.parse_csv_file_dataframe(
    str(csv_path),
    encoding='utf-8',
    na_values={'comment': ['NA', 'NULL']},
    usecols=lambda c: c in {'order_id', 'region', 'qty', 'unit_price', 'status', 'comment'},
    nrows=100000,
)

df.head()

In [ ]:
df.info()

In [ ]:
# Typical EDA step: filter + derived feature.
work = df[df['status'] == 'paid'].copy()
work['revenue'] = work['qty'] * work['unit_price']
work

In [ ]:
summary = (
    work.groupby('region', dropna=False)
    .agg(
        orders=('order_id', 'count'),
        total_revenue=('revenue', 'sum'),
        avg_ticket=('revenue', 'mean'),
    )
    .sort_values('total_revenue', ascending=False)
)
summary

In [ ]:
# Quick chart (optional).
ax = summary['total_revenue'].plot(kind='bar', title='Total Revenue by Region')
ax.set_xlabel('Region')
ax.set_ylabel('Revenue')